# Travaux: Chandra OCR 2 on development set 1 (the GPU build against the laptop's)
The 20 pages of `eval/vlm_pilot/dev_set.json`, read by the original Chandra OCR 2 (fp16, vLLM, two T4s) with the settings of
`kaggle_chandra.ipynb`, so that its readings can be compared with the laptop's MLX 5-bit build on pages with a retyped reference
before any GPU reading is used. Inputs: the datasets **travaux-scans** and **travaux-devset** (the queue of the 20 pages and the
one PDF not in travaux-scans). Output: `results_dev.zip`, copied into `data/ocr/<doc>/chandra-gpu/` on the laptop.

In [ ]:
# vLLM built for CUDA 12.9 (Chandra OCR 2 is a Qwen3.5 model, so a recent vLLM is needed); after it, Kaggle's torchaudio and
# torchcodec are removed: they are built for another PyTorch and transformers / vLLM import them on start (audio only)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" pymupdf openai --extra-index-url https://download.pytorch.org/whl/cu129 2>&1 | grep -v "requires\|incompatible" | tail -3
!pip uninstall -y -q torchaudio torchcodec
!python -c "import torch, vllm; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpus', torch.cuda.device_count(), '| vllm', vllm.__version__)"
import glob, os
ROOT = os.path.dirname(glob.glob('/kaggle/input/**/chandra_queue_remaining.json', recursive=True)[0])
print('code and scans in', ROOT, '|', len(glob.glob(ROOT + '/raw/*.pdf')), 'PDFs')
# the client as committed after the first run waits 30 min for an answer (the dataset's copy has the OpenAI client's 600 s default,
# which timed out 25 of 1,986 pages): a working copy of the code with that one change
CODE = '/kaggle/working/code'
!rm -rf {CODE} && mkdir -p {CODE} && cp -r {ROOT}/pipeline {ROOT}/eval {CODE}/
!grep -q 'timeout=' {CODE}/pipeline/hpc/chandra_client.py || sed -i 's/api_key="local")/api_key="local", timeout=1800)/' {CODE}/pipeline/hpc/chandra_client.py
!grep -c 'timeout=' {CODE}/pipeline/hpc/chandra_client.py

In [ ]:
# the model server: fp16 (the T4 has no bf16), split over the two GPUs; eager mode, no custom all-reduce, NCCL P2P off (the
# settings that started on Kaggle's T4s, 17 Oct 2026); its own process group, so an interrupted cell does not kill it. The first
# start compiles Triton kernels for the T4 (ptxas): about 13 minutes with no log lines, which is not a hang.
import os, subprocess, time, urllib.request
env = dict(os.environ, NCCL_P2P_DISABLE='1', NCCL_IB_DISABLE='1')
server = subprocess.Popen(['vllm', 'serve', 'datalab-to/chandra-ocr-2', '--served-model-name', 'chandra',
                           '--dtype', 'half', '--tensor-parallel-size', '2', '--max-model-len', '16384',
                           '--gpu-memory-utilization', '0.92', '--limit-mm-per-prompt', '{"image": 1}',
                           '--enforce-eager', '--disable-custom-all-reduce', '--port', '8000'],
                          stdout=open('/kaggle/working/vllm.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True, env=env)
for i in range(360):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up after', i * 10, 's'); break
    except Exception:
        if server.poll() is not None: print(open('/kaggle/working/vllm.log').read()[-3000:]); raise SystemExit('vLLM stopped')
        time.sleep(10)

In [ ]:
# the scans of both datasets in one folder, then the 20 pages (4 in flight, as on the laptop each page is read alone)
DEV = os.path.dirname(glob.glob('/kaggle/input/**/dev_set1_queue.json', recursive=True)[0])
!mkdir -p /kaggle/working/raw && ln -sf {ROOT}/raw/*.pdf {DEV}/*.pdf /kaggle/working/raw/
!python {CODE}/pipeline/hpc/chandra_client.py --queue {DEV}/dev_set1_queue.json --raw /kaggle/working/raw --out /kaggle/working/results_dev --workers 4
!cut -c1-160 /kaggle/working/results_dev/chandra_run.jsonl

In [ ]:
os.killpg(server.pid, 9)
!rm -rf /kaggle/working/raw /kaggle/working/code && cd /kaggle/working && zip -q -r results_dev.zip results_dev && ls -la results_dev.zip